# 🚀 Kaggle Qwen 2.5 Coder 7B Instruct Server
This notebook loads the **Qwen2.5-Coder-7B-Instruct** model in 4-bit precision and serves it as a FastAPI completions backend. It exposes the API to the public internet using a secure `pyngrok` tunnel.

### ⚙️ Instructions
1. Turn on the **GPU Accelerator** in the notebook settings (select **GPU T4 x2** or **GPU T4 x1**).
2. Add your **Ngrok Auth Token** to Kaggle Secrets:
   - Go to **Add-ons -> Secrets** in the top menu.
   - Add a secret named `NGROK_AUTH_TOKEN` with your token as the value.
   - Make sure the checkbox next to the secret is checked to share it with the notebook.
3. Run all cells in order.

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    torch \
    sentencepiece \
    fastapi \
    uvicorn \
    pyngrok \
    nest_asyncio \
    pydantic

## 🔑 Step 2: Load Secrets and Import Libraries

In [ ]:
import os
import torch
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
PORT = 8000

# Fetch secret token from Kaggle user secrets
NGROK_AUTH_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_AUTH_TOKEN")
    print("✅ Loaded NGROK_AUTH_TOKEN from Kaggle User Secrets.")
except Exception as e:
    print("❌ Failed to load secret from Kaggle User Secrets. Checking environment variables...")
    NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN")

if not NGROK_AUTH_TOKEN:
    print("⚠️ WARNING: NGROK_AUTH_TOKEN is not set. Please add it to Add-ons -> Secrets.")

✅ Loaded NGROK_AUTH_TOKEN from Kaggle User Secrets.


## 🤖 Step 3: Load Model & Tokenizer (4-bit Precision)

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model in 4-bit nf4 precision...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config,
)
model.eval()
print(f"✅ Model loaded successfully onto: {model.device}")

Loading tokenizer...
Loading model in 4-bit nf4 precision...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✅ Model loaded successfully onto: cuda:0


## 🛠️ Step 4: Define API Schemas and Router

In [ ]:
app = FastAPI(title="Kaggle Qwen Coder 7B API")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    messages: list[ChatMessage]
    temperature: float = 0.0
    max_new_tokens: int = 512

class ChatCompletionResponse(BaseModel):
    success: bool
    response: str
    error_message: str

def generate_chat_internal(messages, temperature=0.0, max_new_tokens=512):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)
    
    do_sample = temperature > 0.0
    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
    }
    if do_sample:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = temperature
    else:
        gen_kwargs["do_sample"] = False
        
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **gen_kwargs
        )
        
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )
    return response

@app.get("/")
def root():
    return {
        "status": "running",
        "model": MODEL_NAME
    }

@app.post("/generate")
def generate(req: ChatCompletionRequest):
    try:
        result = generate_chat_internal(
            messages=[{"role": m.role, "content": m.content} for m in req.messages],
            temperature=req.temperature,
            max_new_tokens=req.max_new_tokens
        )
        return ChatCompletionResponse(
            success=True,
            response=result,
            error_message=""
        )
    except Exception as e:
        return ChatCompletionResponse(
            success=False,
            response="",
            error_message=str(e)
        )

## 🚀 Step 5: Start Public Tunnel and FastAPI Server

In [ ]:
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(PORT)
    print("\n" + "="*80)
    print(f"🚀 SERVER PUBLIC URL: {public_url}")
    print("Copy the URL above and paste it into your Backed/.env as AI_SERVER_URL")
    print("="*80 + "\n")
else:
    print("\n❌ Unable to set up ngrok tunnel without an authentication token.")

# Launch server directly in the notebook's active event loop
import uvicorn
config = uvicorn.Config(app, host="0.0.0.0", port=PORT, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()


🚀 SERVER PUBLIC URL: NgrokTunnel: "https://349d-34-9-82-93.ngrok-free.app" -> "http://localhost:8000"
Copy the URL above and paste it into your Backed/.env as AI_SERVER_URL



INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     39.34.149.180:0 - "GET / HTTP/1.1" 200 OK
INFO:     39.34.149.180:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [58]
